# Intraday PCA-SVM Momentum Query Backtest

This notebook fetches intraday `IRS_RATE` data for `USD-SOFR-1D-Q12STIRT` / `IMM_4xIMM_5`, builds a
PCA-SVM momentum signal (ported from the Matlab `pfBBGDailySVM` pipeline), runs a quick vectorized
sanity check, and then runs the canonical `QueryDrivenBacktest` implementation.

**Algorithm (Matlab port):**
1. EMA−SMA spreads for short/long windows + long-vs-short EMA cross-spreads → feature matrix
2. Feature normalization (StandardScaler) → PCA projection to 2 components
3. Backward-looking momentum z-score → labels {−1, 0, +1}
4. SVM (RBF kernel) trained on initial 70% of data, predictions on remaining 30%

The trade query uses `market_request={"timestamp": "now"}` so the query-driven run requests true
intraday curves.  Data is resampled to 30-min bars to keep SVM training tractable.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pytz

sys.path.append("../../")

from BT.signals import pca_svm_momentum_signal, run_technical_indicator_query_backtest
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

NYC = pytz.timezone("America/New_York")

In [ ]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()
trade_bpv = 100_000.0

start = NYC.localize(datetime.datetime(2026, 1, 4, 18, 0))
end = NYC.localize(datetime.datetime(2026, 3, 27, 17, 0))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_4xIMM_5",
    value=UnifiedValue.IRS_RATE,
)

intraday_df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q],
    freq="1min",
    mdps={"IRS": curve_mdp},
    ignore_cache_miss=True,
)
intraday_df = intraday_df.sort_index()

rate_series = pd.to_numeric(intraday_df.iloc[:, 0], errors="coerce").dropna()
# Resample to 30-min bars for PCA-SVM tractability
rate_series = rate_series.resample("30min").last().dropna()
rate_series.name = "IMM_4xIMM_5"

rate_series

In [ ]:
# PCA-SVM Momentum signal — Matlab pfBBGDailySVM port
# EMA/SMA windows scaled for 30-min bars (~48 bars/day)
signal_result = pca_svm_momentum_signal(
    rate_series,
    ewma_short_spans=[48, 96, 240],       # ~1d, 2d, 5d in 30-min bars
    ewma_long_spans=[480, 720, 960],       # ~10d, 15d, 20d
    n_components=2,                        # Matlab K=2
    svm_c=0.02,                            # Matlab C=0.02
    svm_gamma=0.1,                         # Matlab sigma=0.1
    label_window=48,                       # ~1 day of 30-min bars
    z_threshold=1.75,                      # Matlab labelBreaks=[-inf,-1.75,1.75,inf]
    train_fraction=0.7,                    # Matlab DataSplit=0.7
    retrain_every=48,                      # retrain daily (expanding window)
)

display(signal_result.indicator_frame.tail())
display(
    pd.DataFrame(
        {
            "rate": signal_result.raw_series,
            "desired_position": signal_result.desired_position,
            "execution_position": signal_result.execution_position,
        }
    ).tail()
)

In [ ]:
# Vectorized sanity check only. The canonical result is the query-driven backtest below.
vectorized_position = signal_result.execution_position.fillna(0.0)
vectorized_pnl = -vectorized_position * rate_series.diff().fillna(0.0) * trade_bpv * 100.0
vectorized_cumulative_pnl = vectorized_pnl.cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
vectorized_cumulative_pnl.plot(ax=ax, title="PCA-SVM Momentum Vectorized Sanity Check")
ax.set_ylabel("Approx PnL")
plt.show()

In [ ]:
# Keep timestamp="now" so the query-driven backtest requests true intraday curves.
# For BARCHART_STIRF-RL, single-request pricing consults CurveStore first.
def trade_query_factory(target_position, now, info):
    _ = now, info
    return IRSwapQuery(
        curve="USD-SOFR-1D-Q12STIRT",
        tenor="IMM_4xIMM_5",
        value=IRSwapValue.NPV,
        market_request={"timestamp": "now"},
        structure_kwargs={"bpv": trade_bpv * float(target_position)},
        tags=("intraday_pca_svm_momentum_q12stirt",),
    )


query_backtest = run_technical_indicator_query_backtest(
    signal_result,
    trade_query_factory=trade_query_factory,
    mdp=curve_mdp,
    strategy_name="intraday_pca_svm_momentum_q12stirt",
    ignore_cache_miss=True,
    show_progress=True,
)

query_backtest.metrics

In [ ]:
plot_df = pd.DataFrame(
    {
        "rate": signal_result.raw_series,
        "momentum_zscore": signal_result.indicator_frame["momentum_zscore"],
        "label": signal_result.indicator_frame["label"],
        "decision_score": signal_result.indicator_frame["decision_score"],
        "pc_1": signal_result.indicator_frame["pc_1"],
        "pc_2": signal_result.indicator_frame["pc_2"],
        "execution_position": signal_result.execution_position,
    }
).dropna(subset=["rate"])

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
plot_df["rate"].plot(ax=axes[0], title="Q12 STIRT IMM_4xIMM_5 Rate")
plot_df["momentum_zscore"].dropna().plot(ax=axes[1], title="Momentum Z-Score", color="tab:orange")
axes[1].axhline(1.75, color="gray", linestyle="--", alpha=0.5)
axes[1].axhline(-1.75, color="gray", linestyle="--", alpha=0.5)
plot_df["execution_position"].fillna(0.0).plot(ax=axes[2], title="Execution Position", color="black")
plot_df[["pc_1", "pc_2"]].dropna().plot(ax=axes[3], title="PCA Components")
query_backtest.mtm_history.plot(ax=axes[4], title="Query-Driven MTM", color="tab:green")
axes[4].set_ylabel("PnL")
plt.tight_layout()
plt.show()

query_backtest.order_frame.tail()